# Advanced Problems with Solutions: Custom JSON Encoding with `json.JSONEncoder`

This notebook is a problem-driven deep dive into Python's standard-library JSON encoder.

## Goals

You will practice:

- `json.JSONEncoder` and `default()`
- `json.dumps(..., cls=...)` versus `default=...`
- supported versus unsupported Python types
- strict JSON with `allow_nan=False`
- safe dictionary-key policies
- compact, pretty, and deterministic JSON
- `datetime`, `date`, `time`, `Decimal`, `UUID`, `Path`, `Enum`, dataclasses, sets, bytes, complex numbers, and iterators
- `iterencode()` and circular references
- registry-based and `singledispatch` encoder design
- tagged round-trip formats with `object_hook`
- schema versioning, validation, testing, and security

> Best practice used throughout: if your encoder does not recognize an object, use
> `return super().default(obj)`.

## 0. Setup

In [2]:
import base64
import json
from dataclasses import dataclass, fields, is_dataclass
from datetime import date, datetime, time, timezone
from decimal import Decimal
from enum import Enum, IntEnum
from functools import singledispatch
from io import StringIO
from pathlib import Path
from uuid import UUID, uuid4


def assert_valid_json(text):
    return json.loads(text)


def show(obj, **kwargs):
    print(json.dumps(obj, indent=2, ensure_ascii=False, **kwargs))

# Problem 1 — Prove exactly when `default()` is called

Create a tracing encoder and serialize built-in values plus `datetime` and `complex`.

Questions:

1. Which objects reach `default()`?
2. Why can `default()` not globally uppercase ordinary strings?
3. Why is a tuple serialized without custom handling?

In [3]:
class TracingEncoder(json.JSONEncoder):
    def default(self, obj):
        print(f"default() called for {type(obj).__name__}: {obj!r}")

        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, complex):
            return {
                "__type__": "complex",
                "real": obj.real,
                "imag": obj.imag,
            }

        return super().default(obj)


sample = {
    "text": "Python",
    "integer": 42,
    "float": 3.5,
    "boolean": True,
    "none": None,
    "list": [1, 2, 3],
    "tuple": (4, 5, 6),
    "timestamp": datetime(2026, 8, 7, 13, 30, tzinfo=timezone.utc),
    "number": 2 + 7j,
}

encoded = json.dumps(sample, cls=TracingEncoder, indent=2)
print(encoded)

default() called for datetime: datetime.datetime(2026, 8, 7, 13, 30, tzinfo=datetime.timezone.utc)
default() called for complex: (2+7j)
{
  "text": "Python",
  "integer": 42,
  "float": 3.5,
  "boolean": true,
  "none": null,
  "list": [
    1,
    2,
    3
  ],
  "tuple": [
    4,
    5,
    6
  ],
  "timestamp": {
    "__type__": "datetime",
    "value": "2026-08-07T13:30:00+00:00"
  },
  "number": {
    "__type__": "complex",
    "real": 2.0,
    "imag": 7.0
  }
}


## Solution notes

`default()` is a fallback hook. Built-in JSON-supported values are handled before the hook is consulted.

So:

- strings, integers, floats, booleans, `None`, lists, tuples, and dicts do not normally reach `default()`;
- `datetime` and `complex` do;
- tuples are already supported and become JSON arrays;
- transforming ordinary strings requires preprocessing the object graph, not `default()`.

# Problem 2 — Encode `datetime`, `date`, and `time` correctly

Requirements:

- distinguish the three temporal types;
- use ISO 8601 strings;
- keep explicit type tags;
- remember that `datetime` is a subclass of `date`;
- fall back to the base encoder for unsupported types.

In [4]:
class TemporalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat(),
            }

        # Check date after datetime because datetime is a subclass of date.
        if isinstance(obj, date):
            return {
                "__type__": "date",
                "value": obj.isoformat(),
            }

        if isinstance(obj, time):
            return {
                "__type__": "time",
                "value": obj.isoformat(),
            }

        return super().default(obj)


payload = {
    "created_at": datetime(2026, 8, 7, 16, 42, 5, 123456, tzinfo=timezone.utc),
    "due_date": date(2026, 8, 31),
    "alarm": time(7, 15, 30),
}

text = json.dumps(payload, cls=TemporalEncoder, indent=2)
print(text)

parsed = json.loads(text)

assert parsed["created_at"]["__type__"] == "datetime"
assert parsed["due_date"] == {
    "__type__": "date",
    "value": "2026-08-31",
}
assert parsed["alarm"] == {
    "__type__": "time",
    "value": "07:15:30",
}

{
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T16:42:05.123456+00:00"
  },
  "due_date": {
    "__type__": "date",
    "value": "2026-08-31"
  },
  "alarm": {
    "__type__": "time",
    "value": "07:15:30"
  }
}


# Problem 3 — Preserve `Decimal` precision

A `Decimal` may contain more precision than a binary `float`.

Implement:

1. an exact tagged-string strategy;
2. an approximate float strategy;

and prove that the approximate version loses information.

In [5]:
class ExactDecimalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return {
                "__type__": "decimal",
                "value": str(obj),
            }

        return super().default(obj)


class FloatDecimalEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)

        return super().default(obj)


value = Decimal("0.123456789012345678901234567890123456789")

exact_text = json.dumps(
    {"value": value},
    cls=ExactDecimalEncoder,
)

approx_text = json.dumps(
    {"value": value},
    cls=FloatDecimalEncoder,
)

print("exact :", exact_text)
print("approx:", approx_text)

exact_value = json.loads(exact_text)["value"]["value"]
approx_value = json.loads(approx_text)["value"]

assert exact_value == str(value)
assert Decimal(str(approx_value)) != value

exact : {"value": {"__type__": "decimal", "value": "0.123456789012345678901234567890123456789"}}
approx: {"value": 0.12345678901234568}


## Best practice

If precision matters, do not silently convert `Decimal` to `float`.

Tagged strings are larger, but they preserve exact decimal text and make the contract explicit.

# Problem 4 — Encode `UUID`, `Path`, `Enum`, and inspect `IntEnum`

Build one encoder for several common application types.

In [6]:
class Status(Enum):
    PENDING = "pending"
    DONE = "done"


class Priority(IntEnum):
    LOW = 1
    HIGH = 10


class CommonTypesEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, UUID):
            return {
                "__type__": "uuid",
                "value": str(obj),
            }

        if isinstance(obj, Path):
            return {
                "__type__": "path",
                "value": str(obj),
            }

        if isinstance(obj, Enum):
            return {
                "__type__": "enum",
                "enum_class": type(obj).__name__,
                "name": obj.name,
                "value": obj.value,
            }

        return super().default(obj)


record = {
    "id": uuid4(),
    "file": Path("/var/data/report.json"),
    "status": Status.DONE,
    "priority": Priority.HIGH,
}

print(json.dumps(record, cls=CommonTypesEncoder, indent=2))

{
  "id": {
    "__type__": "uuid",
    "value": "11cf6978-07f0-4abd-9876-78961f941483"
  },
  "file": {
    "__type__": "path",
    "value": "\\var\\data\\report.json"
  },
  "status": {
    "__type__": "enum",
    "enum_class": "Status",
    "name": "DONE",
    "value": "done"
  },
  "priority": 10
}


## Important `IntEnum` subtlety

`IntEnum` is also an `int`. The base JSON encoder may serialize it as a number before `default()` is needed.

This is another example of the rule:

> You cannot rely on `default()` to replace the encoding of a type already accepted by the base encoder.

# Problem 5 — Serialize dataclasses without leaking implementation details

Requirements:

- detect dataclass instances;
- serialize declared dataclass fields;
- add a type tag;
- do not blindly use `obj.__dict__`;
- allow nested custom values to recurse through the encoder.

In [7]:
@dataclass
class Address:
    city: str
    country: str


@dataclass
class Customer:
    customer_id: UUID
    name: str
    registered_at: datetime
    address: Address


class DataclassEncoder(json.JSONEncoder):
    def default(self, obj):
        if is_dataclass(obj) and not isinstance(obj, type):
            values = {
                field.name: getattr(obj, field.name)
                for field in fields(obj)
            }

            return {
                "__type__": "dataclass",
                "class": type(obj).__name__,
                "fields": values,
            }

        if isinstance(obj, UUID):
            return {
                "__type__": "uuid",
                "value": str(obj),
            }

        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat(),
            }

        return super().default(obj)


customer = Customer(
    customer_id=uuid4(),
    name="Ada",
    registered_at=datetime(
        2026, 8, 7, 12, 0, tzinfo=timezone.utc
    ),
    address=Address(
        city="Sofia",
        country="Bulgaria",
    ),
)

print(json.dumps(customer, cls=DataclassEncoder, indent=2))

{
  "__type__": "dataclass",
  "class": "Customer",
  "fields": {
    "customer_id": {
      "__type__": "uuid",
      "value": "4345280a-6af2-46b0-8fa7-a8d6e3cbdff0"
    },
    "name": "Ada",
    "registered_at": {
      "__type__": "datetime",
      "value": "2026-08-07T12:00:00+00:00"
    },
    "address": {
      "__type__": "dataclass",
      "class": "Address",
      "fields": {
        "city": "Sofia",
        "country": "Bulgaria"
      }
    }
  }
}


## Why not `__dict__`?

`__dict__` may contain caches, private attributes, implementation details, or helper objects that are not part of the serialized contract.

Declared fields are usually a safer starting point.

# Problem 6 — Deterministic set encoding

A JSON array can represent set elements, but plain `list(my_set)` is not a stable contract.

Encode `set` and `frozenset` deterministically.

In [8]:
class SetEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, set):
            return {
                "__type__": "set",
                "items": sorted(obj),
            }

        if isinstance(obj, frozenset):
            return {
                "__type__": "frozenset",
                "items": sorted(obj),
            }

        return super().default(obj)


payload = {
    "permissions": {"write", "read", "delete"},
    "immutable_ids": frozenset({9, 2, 5}),
}

first = json.dumps(
    payload,
    cls=SetEncoder,
    sort_keys=True,
)

second = json.dumps(
    payload,
    cls=SetEncoder,
    sort_keys=True,
)

print(first)
assert first == second

{"immutable_ids": {"__type__": "frozenset", "items": [2, 5, 9]}, "permissions": {"__type__": "set", "items": ["delete", "read", "write"]}}


## Edge case

`sorted(obj)` fails if elements are not mutually orderable. For custom objects, define a domain-specific sort key or reject the set explicitly.

# Problem 7 — Encode binary data with Base64

JSON has no binary type. Encode `bytes` with:

- a type tag;
- an encoding tag;
- Base64 text;

then round-trip the bytes manually.

In [9]:
class BytesEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, bytes):
            return {
                "__type__": "bytes",
                "encoding": "base64",
                "data": base64.b64encode(obj).decode("ascii"),
            }

        return super().default(obj)


raw = b"\x00\x01\x02hello\xff"

text = json.dumps(
    {"blob": raw},
    cls=BytesEncoder,
    indent=2,
)

print(text)

node = json.loads(text)["blob"]
restored = base64.b64decode(node["data"])

assert node["encoding"] == "base64"
assert restored == raw

{
  "blob": {
    "__type__": "bytes",
    "encoding": "base64",
    "data": "AAECaGVsbG//"
  }
}


# Problem 8 — Enforce strict JSON

Python permits `NaN`, `Infinity`, and `-Infinity` by default.

Show the permissive behavior, then reject those values with `allow_nan=False`.

In [10]:
non_finite = {
    "nan": float("nan"),
    "positive_infinity": float("inf"),
    "negative_infinity": float("-inf"),
}

print("Default behavior:")
print(json.dumps(non_finite))

try:
    json.dumps(non_finite, allow_nan=False)
except ValueError as exc:
    print("Strict JSON rejected the payload:", exc)

Default behavior:
{"nan": NaN, "positive_infinity": Infinity, "negative_infinity": -Infinity}
Strict JSON rejected the payload: Out of range float values are not JSON compliant: nan


In [11]:
class StrictJSONEncoder(json.JSONEncoder):
    def __init__(self, *args, **kwargs):
        kwargs["allow_nan"] = False
        super().__init__(*args, **kwargs)


try:
    json.dumps(
        non_finite,
        cls=StrictJSONEncoder,
    )
except ValueError as exc:
    print("StrictJSONEncoder rejected the payload:", exc)

StrictJSONEncoder rejected the payload: Out of range float values are not JSON compliant: nan


## Key insight

`default()` cannot intercept ordinary floats, including `NaN` and infinity, because floats are built-in supported values.

Strictness belongs in encoder configuration (`allow_nan=False`) or preprocessing.

# Problem 9 — Avoid silent data loss from `skipkeys=True`

Compare:

- normal failure;
- `skipkeys=True`;
- explicit key normalization.

In [12]:
source = {
    "name": "example",
    10: "integer-key",
    3.5: "float-key",
    (1, 2): "tuple-key",
}

try:
    print(json.dumps(source))
except TypeError as exc:
    print("Default behavior:", exc)

print("\nWith skipkeys=True:")
print(json.dumps(source, skipkeys=True, indent=2))

Default behavior: keys must be str, int, float, bool or None, not tuple

With skipkeys=True:
{
  "name": "example",
  "10": "integer-key",
  "3.5": "float-key"
}


In [13]:
def normalize_keys(mapping):
    normalized = {}

    for key, value in mapping.items():
        if isinstance(key, str):
            json_key = key
        elif isinstance(key, int):
            json_key = f"int:{key}"
        elif isinstance(key, float):
            json_key = f"float:{key!r}"
        else:
            raise TypeError(
                f"Unsupported dictionary key type: {type(key).__name__}"
            )

        if json_key in normalized:
            raise ValueError(
                f"Key collision after normalization: {json_key!r}"
            )

        normalized[json_key] = value

    return normalized


safe = normalize_keys({
    "name": "example",
    10: "integer-key",
    3.5: "float-key",
})

print(json.dumps(safe, indent=2, sort_keys=True))

{
  "float:3.5": "float-key",
  "int:10": "integer-key",
  "name": "example"
}


## Best practice

`skipkeys=True` may make serialization succeed by deleting entries.

In data-sensitive systems, explicit failure or explicit normalization is usually safer.

# Problem 10 — Build an application serialization policy

Create a custom encoder that enforces:

- readable Unicode;
- strict JSON;
- sorted keys;

while allowing callers to choose formatting such as indentation.

In [14]:
class AppJSONEncoder(json.JSONEncoder):
    def __init__(self, *args, **kwargs):
        kwargs["ensure_ascii"] = False
        kwargs["allow_nan"] = False
        kwargs["sort_keys"] = True
        super().__init__(*args, **kwargs)

    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()

        if isinstance(obj, UUID):
            return str(obj)

        return super().default(obj)


data = {
    "message": "Здравей, JSON",
    "time": datetime(
        2026, 8, 7, 15, 0, tzinfo=timezone.utc
    ),
    "id": uuid4(),
}

print(json.dumps(
    data,
    cls=AppJSONEncoder,
    indent=2,
))

{
  "id": "047dae57-d365-48ea-9920-2fbcd8e1f905",
  "message": "Здравей, JSON",
  "time": "2026-08-07T15:00:00+00:00"
}


## Alternative design: wrapper function

A helper function can make defaults explicit while allowing controlled overrides.

In [15]:
def app_dumps(obj, **kwargs):
    options = {
        "ensure_ascii": False,
        "allow_nan": False,
        "sort_keys": True,
        "cls": AppJSONEncoder,
    }

    options.update(kwargs)
    return json.dumps(obj, **options)


print(app_dumps(data, indent=4))
print(app_dumps(data, separators=(",", ":")))

{
    "id": "047dae57-d365-48ea-9920-2fbcd8e1f905",
    "message": "Здравей, JSON",
    "time": "2026-08-07T15:00:00+00:00"
}
{"id":"047dae57-d365-48ea-9920-2fbcd8e1f905","message":"Здравей, JSON","time":"2026-08-07T15:00:00+00:00"}


# Problem 11 — Pretty, compact, and deterministic JSON

Produce three representations and compare lengths.

In [16]:
payload = {
    "language": "Python",
    "versions": [3, 10, 11, 12, 13],
    "active": True,
    "metadata": {
        "creator": "Guido van Rossum",
        "category": "programming language",
    },
}

pretty_text = json.dumps(
    payload,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

compact_text = json.dumps(
    payload,
    ensure_ascii=False,
    separators=(",", ":"),
)

deterministic_text = json.dumps(
    payload,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
    allow_nan=False,
)

print("Pretty length       :", len(pretty_text))
print("Compact length      :", len(compact_text))
print("Deterministic length:", len(deterministic_text))

print("\nPRETTY\n", pretty_text)
print("\nCOMPACT\n", compact_text)
print("\nDETERMINISTIC\n", deterministic_text)

Pretty length       : 198
Compact length      : 138
Deterministic length: 138

PRETTY
 {
  "active": true,
  "language": "Python",
  "metadata": {
    "category": "programming language",
    "creator": "Guido van Rossum"
  },
  "versions": [
    3,
    10,
    11,
    12,
    13
  ]
}

COMPACT
 {"language":"Python","versions":[3,10,11,12,13],"active":true,"metadata":{"creator":"Guido van Rossum","category":"programming language"}}

DETERMINISTIC
 {"active":true,"language":"Python","metadata":{"category":"programming language","creator":"Guido van Rossum"},"versions":[3,10,11,12,13]}


## Important nuance

`sort_keys=True` plus compact separators is useful for tests and cache keys, but it is not a full formal JSON canonicalization standard.

Do not use it for cryptographic signatures unless your protocol defines exactly those serialization rules.

# Problem 12 — Encode arbitrary iterators

Support generators and `range` objects by converting arbitrary iterables to lists.

In [17]:
class IterableEncoder(json.JSONEncoder):
    def default(self, obj):
        try:
            iterator = iter(obj)
        except TypeError:
            return super().default(obj)
        else:
            return list(iterator)


def squares(n):
    for i in range(n):
        yield i * i


text = json.dumps(
    {
        "squares": squares(8),
        "range": range(5),
    },
    cls=IterableEncoder,
    indent=2,
)

print(text)

{
  "squares": [
    0,
    1,
    4,
    9,
    16,
    25,
    36,
    49
  ],
  "range": [
    0,
    1,
    2,
    3,
    4
  ]
}


## Caveat

Materializing an iterator consumes it and can use large amounts of memory.

For huge streams, consider `iterencode()`, manual streaming, or JSON Lines/NDJSON.

# Problem 13 — Stream JSON with `iterencode()`

Inspect encoder chunks and write them incrementally to a text stream.

In [18]:
largeish = {
    "numbers": list(range(20)),
    "message": "stream me",
}

encoder = json.JSONEncoder(
    ensure_ascii=False,
    separators=(",", ":"),
)

chunks = list(encoder.iterencode(largeish))

print("Chunk count:", len(chunks))
print("First chunks:", chunks[:10])

buffer = StringIO()

for chunk in encoder.iterencode(largeish):
    buffer.write(chunk)

streamed_text = buffer.getvalue()

assert json.loads(streamed_text) == largeish
print(streamed_text)

Chunk count: 29
First chunks: ['{', '"numbers"', ':', '[0', ',1', ',2', ',3', ',4', ',5', ',6']
{"numbers":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19],"message":"stream me"}


# Problem 14 — Circular-reference behavior

Create a circular list. Compare default circular checking with `check_circular=False`.

In [19]:
circular = []
circular.append(circular)

try:
    json.dumps(circular)
except ValueError as exc:
    print("Circular check enabled:", type(exc).__name__, exc)

try:
    json.dumps(
        circular,
        check_circular=False,
    )
except (RecursionError, OverflowError) as exc:
    print("Circular check disabled:", type(exc).__name__, exc)

Circular check enabled: ValueError Circular reference detected
Circular check disabled: RecursionError maximum recursion depth exceeded while encoding a JSON object


## Best practice

Keep `check_circular=True` unless you have a measured reason to disable it and can guarantee acyclic data.

# Problem 15 — Replace a giant `if/elif` chain with a registry

Requirements:

- register `(type, serializer)` pairs;
- first matching type wins;
- support subclasses with `isinstance`;
- fall back to `super().default()`.

In [20]:
class RegistryJSONEncoder(json.JSONEncoder):
    _registry = []

    @classmethod
    def register(cls, python_type, serializer):
        cls._registry.append(
            (python_type, serializer)
        )

    def default(self, obj):
        for python_type, serializer in self._registry:
            if isinstance(obj, python_type):
                return serializer(obj)

        return super().default(obj)


def encode_datetime(value):
    return {
        "__type__": "datetime",
        "value": value.isoformat(),
    }


def encode_decimal(value):
    return {
        "__type__": "decimal",
        "value": str(value),
    }


def encode_uuid(value):
    return {
        "__type__": "uuid",
        "value": str(value),
    }


def encode_path(value):
    return {
        "__type__": "path",
        "value": str(value),
    }


RegistryJSONEncoder.register(datetime, encode_datetime)
RegistryJSONEncoder.register(Decimal, encode_decimal)
RegistryJSONEncoder.register(UUID, encode_uuid)
RegistryJSONEncoder.register(Path, encode_path)


payload = {
    "when": datetime(
        2026, 8, 7, 16, 42, tzinfo=timezone.utc
    ),
    "price": Decimal("1234.567890123456789"),
    "id": uuid4(),
    "file": Path("/tmp/data.json"),
}

print(json.dumps(
    payload,
    cls=RegistryJSONEncoder,
    indent=2,
))

{
  "when": {
    "__type__": "datetime",
    "value": "2026-08-07T16:42:00+00:00"
  },
  "price": {
    "__type__": "decimal",
    "value": "1234.567890123456789"
  },
  "id": {
    "__type__": "uuid",
    "value": "5ad3ab94-116c-483d-867d-2222c19a17db"
  },
  "file": {
    "__type__": "path",
    "value": "\\tmp\\data.json"
  }
}


# Problem 16 — Use `singledispatch` for serializer dispatch

Move type dispatch out of the encoder class.

In [21]:
@singledispatch
def to_json_compatible(obj):
    raise TypeError(
        f"Object of type {type(obj).__name__} is not JSON serializable"
    )


@to_json_compatible.register
def _(obj: datetime):
    return {
        "__type__": "datetime",
        "value": obj.isoformat(),
    }


@to_json_compatible.register
def _(obj: Decimal):
    return {
        "__type__": "decimal",
        "value": str(obj),
    }


@to_json_compatible.register
def _(obj: UUID):
    return {
        "__type__": "uuid",
        "value": str(obj),
    }


@to_json_compatible.register
def _(obj: Path):
    return {
        "__type__": "path",
        "value": str(obj),
    }


class DispatchEncoder(json.JSONEncoder):
    def default(self, obj):
        try:
            return to_json_compatible(obj)
        except TypeError:
            return super().default(obj)


example = {
    "timestamp": datetime(
        2026, 8, 7, 8, 30, tzinfo=timezone.utc
    ),
    "amount": Decimal("99.9900000000000000001"),
    "job_id": uuid4(),
    "output": Path("reports/final.json"),
}

print(json.dumps(
    example,
    cls=DispatchEncoder,
    indent=2,
))

{
  "timestamp": {
    "__type__": "datetime",
    "value": "2026-08-07T08:30:00+00:00"
  },
  "amount": {
    "__type__": "decimal",
    "value": "99.9900000000000000001"
  },
  "job_id": {
    "__type__": "uuid",
    "value": "2bb07256-ef3e-4dd5-a0c7-b0a0177c8bee"
  },
  "output": {
    "__type__": "path",
    "value": "reports\\final.json"
  }
}


# Problem 17 — Tagged round-trip encoding and decoding

Support reversible representations for:

- `datetime`
- `Decimal`
- `UUID`
- `complex`
- `bytes`
- `set`

Use an allowlisted `object_hook`.

In [22]:
class TaggedEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, Decimal):
            return {
                "__type__": "decimal",
                "value": str(obj),
            }

        if isinstance(obj, UUID):
            return {
                "__type__": "uuid",
                "value": str(obj),
            }

        if isinstance(obj, complex):
            return {
                "__type__": "complex",
                "real": obj.real,
                "imag": obj.imag,
            }

        if isinstance(obj, bytes):
            return {
                "__type__": "bytes",
                "encoding": "base64",
                "data": base64.b64encode(obj).decode("ascii"),
            }

        if isinstance(obj, set):
            return {
                "__type__": "set",
                "items": sorted(obj),
            }

        return super().default(obj)


def tagged_object_hook(obj):
    tag = obj.get("__type__")

    if tag is None:
        return obj

    if tag == "datetime":
        return datetime.fromisoformat(obj["value"])

    if tag == "decimal":
        return Decimal(obj["value"])

    if tag == "uuid":
        return UUID(obj["value"])

    if tag == "complex":
        return complex(
            obj["real"],
            obj["imag"],
        )

    if tag == "bytes":
        if obj.get("encoding") != "base64":
            raise ValueError("Unsupported bytes encoding")

        return base64.b64decode(obj["data"])

    if tag == "set":
        return set(obj["items"])

    # Unknown tags remain plain dictionaries.
    return obj

In [23]:
original = {
    "at": datetime(
        2026, 8, 7, 16, 42, tzinfo=timezone.utc
    ),
    "amount": Decimal("12.345678901234567890"),
    "id": uuid4(),
    "z": 4 - 9j,
    "blob": b"\x01hello\xfe",
    "roles": {"admin", "editor"},
}

text = json.dumps(
    original,
    cls=TaggedEncoder,
    sort_keys=True,
)

restored = json.loads(
    text,
    object_hook=tagged_object_hook,
)

print(text)
print(restored)

assert restored["at"] == original["at"]
assert restored["amount"] == original["amount"]
assert restored["id"] == original["id"]
assert restored["z"] == original["z"]
assert restored["blob"] == original["blob"]
assert restored["roles"] == original["roles"]

{"amount": {"__type__": "decimal", "value": "12.345678901234567890"}, "at": {"__type__": "datetime", "value": "2026-08-07T16:42:00+00:00"}, "blob": {"__type__": "bytes", "data": "AWhlbGxv/g==", "encoding": "base64"}, "id": {"__type__": "uuid", "value": "b23c4028-46f1-4f86-b3f9-3ee99e4cce13"}, "roles": {"__type__": "set", "items": ["admin", "editor"]}, "z": {"__type__": "complex", "imag": -9.0, "real": 4.0}}
{'amount': Decimal('12.345678901234567890'), 'at': datetime.datetime(2026, 8, 7, 16, 42, tzinfo=datetime.timezone.utc), 'blob': b'\x01hello\xfe', 'id': UUID('b23c4028-46f1-4f86-b3f9-3ee99e4cce13'), 'roles': {'editor', 'admin'}, 'z': (4-9j)}


## Security rule

Do not dynamically import and instantiate arbitrary classes from type names contained in untrusted JSON.

Use a small explicit allowlist of tags and decoder functions.

# Problem 18 — Version a long-lived domain schema

Encode a `Money` object with:

- stable type tag;
- explicit schema version;
- explicit fields.

In [24]:
@dataclass
class Money:
    amount: Decimal
    currency: str


class VersionedDomainEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Money):
            return {
                "__type__": "money",
                "__version__": 1,
                "amount": str(obj.amount),
                "currency": obj.currency,
            }

        return super().default(obj)


invoice = {
    "subtotal": Money(
        Decimal("125.50"),
        "EUR",
    ),
    "tax": Money(
        Decimal("25.10"),
        "EUR",
    ),
}

print(json.dumps(
    invoice,
    cls=VersionedDomainEncoder,
    indent=2,
))

{
  "subtotal": {
    "__type__": "money",
    "__version__": 1,
    "amount": "125.50",
    "currency": "EUR"
  },
  "tax": {
    "__type__": "money",
    "__version__": 1,
    "amount": "25.10",
    "currency": "EUR"
  }
}


# Problem 19 — Transform supported strings by preprocessing

Redact values under keys named:

- `password`
- `token`
- `api_key`

This cannot be solved reliably with `default()` because ordinary strings are already supported.

In [25]:
SENSITIVE_KEYS = {
    "password",
    "token",
    "api_key",
}


def redact_for_json(obj):
    if isinstance(obj, dict):
        result = {}

        for key, value in obj.items():
            if key in SENSITIVE_KEYS:
                result[key] = "***REDACTED***"
            else:
                result[key] = redact_for_json(value)

        return result

    if isinstance(obj, list):
        return [
            redact_for_json(item)
            for item in obj
        ]

    if isinstance(obj, tuple):
        return [
            redact_for_json(item)
            for item in obj
        ]

    return obj


request = {
    "user": "alice",
    "password": "do-not-log-this",
    "nested": {
        "token": "secret-token",
        "values": [
            1,
            2,
            {"api_key": "xyz"},
        ],
    },
}

safe_request = redact_for_json(request)
text = json.dumps(safe_request, indent=2)

print(text)

assert "do-not-log-this" not in text
assert "secret-token" not in text
assert "xyz" not in text

{
  "user": "alice",
  "password": "***REDACTED***",
  "nested": {
    "token": "***REDACTED***",
    "values": [
      1,
      2,
      {
        "api_key": "***REDACTED***"
      }
    ]
  }
}


# Problem 20 — Production-style tests

Build an encoder policy and test:

- Unicode;
- strict floats;
- unsupported objects;
- temporal format;
- determinism;
- JSON validity.

In [26]:
class ProductionEncoder(json.JSONEncoder):
    def __init__(self, *args, **kwargs):
        kwargs["ensure_ascii"] = False
        kwargs["allow_nan"] = False
        super().__init__(*args, **kwargs)

    def default(self, obj):
        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, Decimal):
            return {
                "__type__": "decimal",
                "value": str(obj),
            }

        if isinstance(obj, UUID):
            return {
                "__type__": "uuid",
                "value": str(obj),
            }

        return super().default(obj)


class Unsupported:
    pass


# Unicode remains readable.
text = json.dumps(
    {"message": "Здравей 🌍"},
    cls=ProductionEncoder,
)

assert "Здравей" in text
assert "🌍" in text


# NaN is rejected.
try:
    json.dumps(
        {"bad": float("nan")},
        cls=ProductionEncoder,
    )
except ValueError:
    pass
else:
    raise AssertionError("NaN should have been rejected")


# Unsupported custom object raises TypeError.
try:
    json.dumps(
        {"x": Unsupported()},
        cls=ProductionEncoder,
    )
except TypeError:
    pass
else:
    raise AssertionError("Unsupported object should raise TypeError")


# Datetime format is explicit.
dt = datetime(
    2026, 8, 7, 16, 42, tzinfo=timezone.utc
)

node = json.loads(
    json.dumps(
        {"dt": dt},
        cls=ProductionEncoder,
    )
)["dt"]

assert node == {
    "__type__": "datetime",
    "value": "2026-08-07T16:42:00+00:00",
}


# Deterministic output.
payload = {
    "b": 2,
    "a": 1,
    "amount": Decimal("10.00"),
}

first = json.dumps(
    payload,
    cls=ProductionEncoder,
    sort_keys=True,
    separators=(",", ":"),
)

second = json.dumps(
    payload,
    cls=ProductionEncoder,
    sort_keys=True,
    separators=(",", ":"),
)

assert first == second
assert_valid_json(first)

print("All production encoder tests passed.")

All production encoder tests passed.


# Problem 21 — Final advanced encoder

Build a reusable encoder supporting:

- strict JSON;
- readable Unicode;
- `datetime`, `date`, `time`;
- `Decimal`;
- `UUID`;
- `Path`;
- `complex`;
- `bytes`;
- deterministic `set`;
- dataclass instances;

plus helper functions for pretty, compact, and deterministic output.

In [27]:
class AdvancedJSONEncoder(json.JSONEncoder):
    def __init__(self, *args, **kwargs):
        kwargs["ensure_ascii"] = False
        kwargs["allow_nan"] = False
        super().__init__(*args, **kwargs)

    def default(self, obj):
        if isinstance(obj, datetime):
            return {
                "__type__": "datetime",
                "value": obj.isoformat(),
            }

        if isinstance(obj, date):
            return {
                "__type__": "date",
                "value": obj.isoformat(),
            }

        if isinstance(obj, time):
            return {
                "__type__": "time",
                "value": obj.isoformat(),
            }

        if isinstance(obj, Decimal):
            return {
                "__type__": "decimal",
                "value": str(obj),
            }

        if isinstance(obj, UUID):
            return {
                "__type__": "uuid",
                "value": str(obj),
            }

        if isinstance(obj, Path):
            return {
                "__type__": "path",
                "value": str(obj),
            }

        if isinstance(obj, complex):
            return {
                "__type__": "complex",
                "real": obj.real,
                "imag": obj.imag,
            }

        if isinstance(obj, bytes):
            return {
                "__type__": "bytes",
                "encoding": "base64",
                "data": base64.b64encode(obj).decode("ascii"),
            }

        if isinstance(obj, set):
            try:
                items = sorted(obj)
            except TypeError:
                raise TypeError(
                    "Set elements must be sortable for deterministic encoding"
                ) from None

            return {
                "__type__": "set",
                "items": items,
            }

        if is_dataclass(obj) and not isinstance(obj, type):
            return {
                "__type__": "dataclass",
                "class": type(obj).__name__,
                "fields": {
                    field.name: getattr(obj, field.name)
                    for field in fields(obj)
                },
            }

        return super().default(obj)


def dumps_pretty(obj):
    return json.dumps(
        obj,
        cls=AdvancedJSONEncoder,
        indent=2,
        sort_keys=True,
    )


def dumps_compact(obj):
    return json.dumps(
        obj,
        cls=AdvancedJSONEncoder,
        separators=(",", ":"),
    )


def dumps_deterministic(obj):
    return json.dumps(
        obj,
        cls=AdvancedJSONEncoder,
        sort_keys=True,
        separators=(",", ":"),
    )

In [28]:
@dataclass
class Event:
    event_id: UUID
    occurred_at: datetime
    amount: Decimal
    file: Path


event = Event(
    event_id=uuid4(),
    occurred_at=datetime(
        2026, 8, 7, 16, 42, tzinfo=timezone.utc
    ),
    amount=Decimal("123.4500"),
    file=Path("/srv/events/42.json"),
)

complex_payload = {
    "event": event,
    "birthday": date(1990, 1, 2),
    "alarm": time(6, 45),
    "vector": 3 + 4j,
    "binary": b"abc\x00\xff",
    "roles": {"admin", "reviewer"},
    "unicode": "Пример 🌍",
}

pretty_output = dumps_pretty(complex_payload)
compact_output = dumps_compact(complex_payload)
deterministic_1 = dumps_deterministic(complex_payload)
deterministic_2 = dumps_deterministic(complex_payload)

print(pretty_output)

assert_valid_json(pretty_output)
assert_valid_json(compact_output)
assert deterministic_1 == deterministic_2
assert len(compact_output) <= len(pretty_output)

print(
    "\nPretty bytes:",
    len(pretty_output.encode("utf-8")),
)

print(
    "Compact bytes:",
    len(compact_output.encode("utf-8")),
)

print("Advanced encoder checks passed.")

{
  "alarm": {
    "__type__": "time",
    "value": "06:45:00"
  },
  "binary": {
    "__type__": "bytes",
    "data": "YWJjAP8=",
    "encoding": "base64"
  },
  "birthday": {
    "__type__": "date",
    "value": "1990-01-02"
  },
  "event": {
    "__type__": "dataclass",
    "class": "Event",
    "fields": {
      "amount": {
        "__type__": "decimal",
        "value": "123.4500"
      },
      "event_id": {
        "__type__": "uuid",
        "value": "772bdf2e-0497-4290-9b82-837d3835ff7e"
      },
      "file": {
        "__type__": "path",
        "value": "\\srv\\events\\42.json"
      },
      "occurred_at": {
        "__type__": "datetime",
        "value": "2026-08-07T16:42:00+00:00"
      }
    }
  },
  "roles": {
    "__type__": "set",
    "items": [
      "admin",
      "reviewer"
    ]
  },
  "unicode": "Пример 🌍",
  "vector": {
    "__type__": "complex",
    "imag": 4.0,
    "real": 3.0
  }
}

Pretty bytes: 932
Compact bytes: 631
Advanced encoder checks passed.


# Problem 22 — Predict failure modes

Predict each result before running.

1. Unsupported object.
2. Non-finite float.
3. Complex dictionary key.
4. Heterogeneous set.
5. Ordinary built-in string.

In [29]:
cases = [
    (
        "unsupported object",
        lambda: json.dumps(
            {"x": object()},
            cls=AdvancedJSONEncoder,
        ),
    ),
    (
        "non-finite float",
        lambda: json.dumps(
            {"x": float("inf")},
            cls=AdvancedJSONEncoder,
        ),
    ),
    (
        "complex dictionary key",
        lambda: json.dumps(
            {1 + 2j: "value"},
            cls=AdvancedJSONEncoder,
        ),
    ),
    (
        "heterogeneous set",
        lambda: json.dumps(
            {"x": {1, "two"}},
            cls=AdvancedJSONEncoder,
        ),
    ),
    (
        "built-in string",
        lambda: json.dumps(
            {"x": "hello"},
            cls=AdvancedJSONEncoder,
        ),
    ),
]

for name, operation in cases:
    print(f"\n--- {name} ---")

    try:
        print(operation())
    except Exception as exc:
        print(type(exc).__name__, exc)


--- unsupported object ---
TypeError Object of type object is not JSON serializable

--- non-finite float ---
ValueError Out of range float values are not JSON compliant: inf

--- complex dictionary key ---
TypeError keys must be str, int, float, bool or None, not complex

--- heterogeneous set ---
TypeError Set elements must be sortable for deterministic encoding

--- built-in string ---
{"x": "hello"}


## Expected reasoning

- unsupported object → `TypeError`;
- non-finite float → `ValueError`;
- complex dictionary key → `TypeError`;
- heterogeneous set → `TypeError` from deterministic set policy;
- built-in string → ordinary JSON encoding, with no `default()` call.

# Problem 23 — Reject naive datetimes and normalize to UTC

In [30]:
class UTCDateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            if obj.tzinfo is None or obj.utcoffset() is None:
                raise ValueError(
                    "Naive datetime is not allowed"
                )

            utc_value = obj.astimezone(timezone.utc)

            return {
                "__type__": "datetime",
                "value": utc_value.isoformat(),
            }

        return super().default(obj)


aware = datetime(
    2026, 8, 7, 18, 0, tzinfo=timezone.utc
)

print(json.dumps(
    {"at": aware},
    cls=UTCDateTimeEncoder,
    indent=2,
))

try:
    json.dumps(
        {"at": datetime(2026, 8, 7, 18, 0)},
        cls=UTCDateTimeEncoder,
    )
except ValueError as exc:
    print("Rejected naive datetime:", exc)

{
  "at": {
    "__type__": "datetime",
    "value": "2026-08-07T18:00:00+00:00"
  }
}
Rejected naive datetime: Naive datetime is not allowed


# Problem 24 — Preserve tuple identity by preprocessing

By default, tuples become JSON arrays and decode back as lists.

Create a tagged tuple representation before serialization.

In [31]:
def preserve_tuples(obj):
    if isinstance(obj, tuple):
        return {
            "__type__": "tuple",
            "items": [
                preserve_tuples(item)
                for item in obj
            ],
        }

    if isinstance(obj, list):
        return [
            preserve_tuples(item)
            for item in obj
        ]

    if isinstance(obj, dict):
        return {
            key: preserve_tuples(value)
            for key, value in obj.items()
        }

    return obj


original = {
    "point": (10, 20),
    "nested": [
        (1, 2),
        (3, 4),
    ],
}

prepared = preserve_tuples(original)
print(json.dumps(prepared, indent=2))

{
  "point": {
    "__type__": "tuple",
    "items": [
      10,
      20
    ]
  },
  "nested": [
    {
      "__type__": "tuple",
      "items": [
        1,
        2
      ]
    },
    {
      "__type__": "tuple",
      "items": [
        3,
        4
      ]
    }
  ]
}


## Why preprocessing?

Tuple is already supported by the base encoder, so `default()` does not reliably get a chance to preserve tuple identity.

# Problem 25 — Detect key collisions after normalization

This input is dangerous:

```python
{"1": "string key", 1: "integer key"}
```

Both can become the same JSON object member name.

In [32]:
def stringify_int_keys(mapping):
    normalized = {}

    for key, value in mapping.items():
        if isinstance(key, str):
            new_key = key
        elif isinstance(key, int):
            new_key = str(key)
        else:
            raise TypeError(
                f"Unsupported key: {key!r}"
            )

        if new_key in normalized:
            raise ValueError(
                f"Collision: multiple source keys map to {new_key!r}"
            )

        normalized[new_key] = value

    return normalized


dangerous = {
    "1": "string key",
    1: "integer key",
}

try:
    stringify_int_keys(dangerous)
except ValueError as exc:
    print("Collision detected:", exc)

Collision detected: Collision: multiple source keys map to '1'


# Problem 26 — Path-aware redaction

Redact only these exact paths:

- `request.headers.authorization`
- `request.credentials.token`

Do not redact unrelated keys with the same names.

In [33]:
REDACT_PATHS = {
    ("request", "headers", "authorization"),
    ("request", "credentials", "token"),
}


def redact_paths(obj, path=()):
    if isinstance(obj, dict):
        result = {}

        for key, value in obj.items():
            child_path = path + (key,)

            if child_path in REDACT_PATHS:
                result[key] = "***REDACTED***"
            else:
                result[key] = redact_paths(
                    value,
                    child_path,
                )

        return result

    if isinstance(obj, list):
        return [
            redact_paths(
                item,
                path + (index,),
            )
            for index, item in enumerate(obj)
        ]

    return obj


log_record = {
    "request": {
        "headers": {
            "authorization": "Bearer SECRET",
            "token": "keep-this-token",
        },
        "credentials": {
            "token": "SUPER-SECRET",
        },
    },
}

print(json.dumps(
    redact_paths(log_record),
    indent=2,
))

{
  "request": {
    "headers": {
      "authorization": "***REDACTED***",
      "token": "keep-this-token"
    },
    "credentials": {
      "token": "***REDACTED***"
    }
  }
}


# Problem 27 — Decode multiple schema versions

Support two historical encodings of the same money value.

In [34]:
@dataclass(frozen=True)
class MoneyValue:
    amount: Decimal
    currency: str


def money_object_hook(obj):
    if obj.get("__type__") != "money":
        return obj

    version = obj.get("__version__")

    if version == 1:
        return MoneyValue(
            amount=Decimal(obj["amount"]),
            currency=obj["currency"],
        )

    if version == 2:
        scale = int(obj["scale"])
        minor_units = int(obj["minor_units"])

        amount = (
            Decimal(minor_units)
            / (Decimal(10) ** scale)
        )

        return MoneyValue(
            amount=amount,
            currency=obj["currency"],
        )

    raise ValueError(
        f"Unsupported money schema version: {version!r}"
    )


v1 = (
    '{"__type__":"money","__version__":1,'
    '"amount":"12.50","currency":"EUR"}'
)

v2 = (
    '{"__type__":"money","__version__":2,'
    '"minor_units":1250,"currency":"EUR","scale":2}'
)

m1 = json.loads(
    v1,
    object_hook=money_object_hook,
)

m2 = json.loads(
    v2,
    object_hook=money_object_hook,
)

print(m1)
print(m2)

assert m1 == m2 == MoneyValue(
    Decimal("12.50"),
    "EUR",
)

MoneyValue(amount=Decimal('12.50'), currency='EUR')
MoneyValue(amount=Decimal('12.5'), currency='EUR')


# Problem 28 — Stream a large JSON array without building a giant list

Write values incrementally.

In [35]:
def stream_json_array(values, write):
    write("[")

    first = True

    for value in values:
        if not first:
            write(",")

        write(json.dumps(value))
        first = False

    write("]")


buffer = StringIO()

stream_json_array(
    range(100),
    buffer.write,
)

text = buffer.getvalue()

assert json.loads(text) == list(range(100))
print(text[:120] + "...")

[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,...


# Problem 29 — Benchmark encoding strategies

Compare:

- `default=...`;
- `cls=...`;
- preprocessing + plain `json.dumps()`.

Treat the numbers as local evidence only.

In [36]:
import timeit


benchmark_data = [
    {
        "id": i,
        "name": f"item-{i}",
        "price": Decimal("19.99"),
    }
    for i in range(100)
]


def decimal_default(obj):
    if isinstance(obj, Decimal):
        return str(obj)

    raise TypeError


class DecimalStringEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Decimal):
            return str(obj)

        return super().default(obj)


def preprocess_decimals(obj):
    if isinstance(obj, Decimal):
        return str(obj)

    if isinstance(obj, list):
        return [
            preprocess_decimals(item)
            for item in obj
        ]

    if isinstance(obj, dict):
        return {
            key: preprocess_decimals(value)
            for key, value in obj.items()
        }

    return obj


prepared = preprocess_decimals(
    benchmark_data
)

tests = {
    "default function": lambda: json.dumps(
        benchmark_data,
        default=decimal_default,
    ),
    "encoder class": lambda: json.dumps(
        benchmark_data,
        cls=DecimalStringEncoder,
    ),
    "preprocessed": lambda: json.dumps(
        prepared
    ),
}

for name, fn in tests.items():
    seconds = timeit.timeit(
        fn,
        number=200,
    )

    print(
        f"{name:16s}: "
        f"{seconds:.6f} seconds"
    )

default function: 0.030855 seconds
encoder class   : 0.034484 seconds
preprocessed    : 0.019203 seconds


# Problem 30 — Add business context to serialization errors

Preserve the original exception using exception chaining.

In [37]:
def dumps_invoice(invoice_id, payload):
    try:
        return json.dumps(
            payload,
            cls=AdvancedJSONEncoder,
        )
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Could not serialize invoice {invoice_id}: {exc}"
        ) from exc


try:
    dumps_invoice(
        "INV-2041",
        {"unsupported": object()},
    )
except ValueError as exc:
    print(exc)
    print("Original cause:", repr(exc.__cause__))

Could not serialize invoice INV-2041: Object of type object is not JSON serializable
Original cause: TypeError('Object of type object is not JSON serializable')


# Problem 31 — Stable set encoding for custom objects

A set contains immutable domain objects that are not naturally orderable.

Define a stable sort key.

In [38]:
@dataclass(frozen=True)
class Permission:
    resource: str
    action: str


class PermissionSetEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, set) and all(
            isinstance(item, Permission)
            for item in obj
        ):
            ordered = sorted(
                obj,
                key=lambda p: (
                    p.resource,
                    p.action,
                ),
            )

            return {
                "__type__": "permission_set",
                "items": [
                    {
                        "resource": p.resource,
                        "action": p.action,
                    }
                    for p in ordered
                ],
            }

        return super().default(obj)


permissions = {
    Permission("invoice", "read"),
    Permission("invoice", "write"),
    Permission("customer", "read"),
}

print(json.dumps(
    permissions,
    cls=PermissionSetEncoder,
    indent=2,
))

{
  "__type__": "permission_set",
  "items": [
    {
      "resource": "customer",
      "action": "read"
    },
    {
      "resource": "invoice",
      "action": "read"
    },
    {
      "resource": "invoice",
      "action": "write"
    }
  ]
}


# Problem 32 — Safe decoder allowlist

Do not deserialize arbitrary class names from untrusted JSON.

Use a fixed map of safe tags.

In [39]:
SAFE_DECODERS = {
    "decimal": lambda obj: Decimal(obj["value"]),
    "uuid": lambda obj: UUID(obj["value"]),
    "datetime": lambda obj: datetime.fromisoformat(obj["value"]),
}


def safe_object_hook(obj):
    tag = obj.get("__type__")

    if tag is None:
        return obj

    decoder = SAFE_DECODERS.get(tag)

    if decoder is None:
        return obj

    return decoder(obj)


safe_text = json.dumps({
    "price": {
        "__type__": "decimal",
        "value": "10.25",
    }
})

decoded = json.loads(
    safe_text,
    object_hook=safe_object_hook,
)

print(decoded)

assert decoded["price"] == Decimal("10.25")

{'price': Decimal('10.25')}


# Extra challenge set

Try these independently:

1. Add support for `frozenset` to the final encoder.
2. Reject all naive `datetime` objects globally.
3. Encode `timezone.utc` explicitly.
4. Add a schema version to every tagged custom object.
5. Create a registry that rejects duplicate type registration.
6. Create one registry per encoder subclass instead of sharing one global list.
7. Add exact support for `Decimal("NaN")`, `Decimal("Infinity")`, and `Decimal("-Infinity")` under a documented tagged string policy.
8. Add a maximum collection-size policy before converting iterators to lists.
9. Write a recursive validator that finds unsupported objects before serialization and reports their object path.
10. Produce JSON Lines instead of one giant array.
11. Add unit tests for malformed Base64 data.
12. Add a decoder that rejects unknown schema versions.
13. Write a custom encoder for a tree node while detecting application-level cycles.
14. Compare `json.dump()` to `json.dumps()` using a file-like object.
15. Test how `ensure_ascii=True` changes Unicode output size.

# Final best-practices checklist

1. Use `default()` for types the base encoder does not already support.
2. Fall back with `return super().default(obj)`.
3. Preprocess when you must transform built-in values.
4. Preserve precision intentionally.
5. Prefer timezone-aware datetimes for distributed systems.
6. Use explicit tagged forms when lossless round trips matter.
7. Use `allow_nan=False` when strict JSON matters.
8. Avoid `skipkeys=True` when silent data loss is unacceptable.
9. Normalize keys explicitly and detect collisions.
10. Use `ensure_ascii=False` for readable UTF-8 JSON when appropriate.
11. Use `separators=(",", ":")` for compact payloads.
12. Use `sort_keys=True` for stable ordering when helpful.
13. Do not confuse stable stdlib output with formal canonical JSON.
14. Leave circular-reference checking enabled unless you know the graph is acyclic.
15. Be careful when converting arbitrary iterators to lists.
16. Prefer registry or dispatch patterns as custom type support grows.
17. Version long-lived persisted schemas.
18. Use allowlisted decoding for untrusted JSON.
19. Test edge cases, failure cases, Unicode, precision, strictness, determinism, and round trips.
20. Document your JSON representation as an external contract, not merely an implementation detail.